# Music Annotation Project — Notebook 38
## V8.2i Envelope Relaxation and Perceptual Severity Calibration — Amended

**Runtime:** Google Colab; CPU is sufficient. Run cells in order.

**Status:** Development-only, **single-reviewer exploratory**. This notebook does not retrain, refit or recalibrate V8.2, reinterpret truth, change Notebook 37 decisions, authorise a lower PRINT threshold, freeze a challenger, consume the untouched future validation cohort, or authorise production use.

### Scientific question
Within the frozen Notebook 37 development frontier, how far can D3–D8 be relaxed before genuine-print recovery is outweighed by perceptually material retained annotation, source-group instability or review burden?

### Amendment 38.1
Notebook 37's curated upload manifest is audited but is not treated as if every package-only row were a mandatory analytical source file. Directly resolved mandatory Notebook 38 inputs remain fail-closed. Unresolved or mismatched non-mandatory package rows are retained as explicit audit warnings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pyarrow matplotlib pillow scipy


## 1. Configuration and locked protocol choices


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, itertools, json, math, os, shutil, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None  # Strongly preferred: exact completed Notebook 37 run.

NOTEBOOK_VERSION='38-v8.2i-envelope-relaxation-amended-38.1'
PROTOCOL_VERSION='1.0-single-reviewer-exploratory'
REVIEWER_ID='reviewer_01'; REVIEWER_COUNT=1
ASSIGNMENT_SEED=38001; BOOTSTRAP_SEED=37001; BOOTSTRAP_RESAMPLES=10_000
CALIBRATION_GROUP_FRACTION=0.50
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
D4_POINT=[.10,.15,.20,.25,.35,.50]; D4_UPPER=[.20,.25,.30,.40,.50,.75]
D5_POINT=[.005,.01,.02,.05,.10]; D5_UPPER=[.01,.02,.05,.10,.15]
FP_WEIGHTS=[1,2,3,5,10]; P1_WEIGHTS=[0,1]; P2_WEIGHTS=[5,10,25]; P3_WEIGHTS=[100,250,500]
D7_TP=[100,250,500,750,1000]; D7_PAGES=[5,8,10,15]; D7_GROUPS=[2,3,4,5]

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_envelope_relaxation'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; REVIEW=OUT/'review'; AUDIT=OUT/'audit'; VISUALS=OUT/'visuals'; REF=OUT/'reference'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,VISUALS,REF]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers, durable failure records and protocol lock


In [ ]:
def utc_now(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def atomic_text(path,text):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+'.tmp'); tmp.write_text(text,encoding='utf-8'); os.replace(tmp,path)
def atomic_json(path,obj): atomic_text(path,json.dumps(obj,indent=2,sort_keys=True,default=str))
def atomic_csv(df,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+'.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)
def fail_closed(stage,message,details=None):
    atomic_json(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':message,'details':details or {},'timestamp_utc':utc_now(),'single_reviewer_exploratory':True})
    raise RuntimeError(f'{stage}: {message}')
def find_col(df,names,required=True):
    lookup={str(c).lower():c for c in df.columns}
    for n in names:
        if n.lower() in lookup:return lookup[n.lower()]
    if required: raise KeyError(f'None of {names} found; columns={list(df.columns)}')
def valid_image(path):
    try:
        with Image.open(path) as im: im.verify()
        return True
    except Exception:return False
def record_hash(d):return hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest()

def make_protocol_lock():
    return {
      'notebook_version':NOTEBOOK_VERSION,'protocol_version':PROTOCOL_VERSION,'created_utc':utc_now(),
      'status':'development-only-single-reviewer-exploratory','reviewer_count':1,'reviewer_id':REVIEWER_ID,
      'may_nominate_revised_envelope_for_independent_test':False,'production_authority':False,
      'untouched_future_validation_used':False,'thresholds':THRESHOLDS,'assignment_seed':ASSIGNMENT_SEED,
      'bootstrap_seed':BOOTSTRAP_SEED,'bootstrap_resamples':BOOTSTRAP_RESAMPLES,
      'calibration_group_fraction':CALIBRATION_GROUP_FRACTION,'registered_utility':'TP-5*FP-500*C',
      'D4_point_ceilings':D4_POINT,'D4_upper_ceilings':D4_UPPER,'D5_point_ceilings':D5_POINT,'D5_upper_ceilings':D5_UPPER,
      'D6_weights':{'FP':FP_WEIGHTS,'P0':[0],'P1':P1_WEIGHTS,'P2':P2_WEIGHTS,'P3':P3_WEIGHTS,'P4':'hard_fail'},
      'D7_values':{'true_print':D7_TP,'benefiting_pages':D7_PAGES,'benefiting_source_groups':D7_GROUPS,'benefit':['B1+','B2+','B3+']},
      'sparse_hard_negative':{'role':'explicitly sparse training-role hard negatives','D9':'not validated without separately scored held-out sparse evidence'},
      'hard_stops':['D1 integrity failure','D2 immutability failure','P4','repeated P3 mechanism','truth exposure before first-pass lock','incomplete mandatory review']}
LOCK=make_protocol_lock(); atomic_json(OUT/'protocol_lock.json',LOCK); print(json.dumps(LOCK,indent=2))


## 3. Resolve mandatory Notebook 37 inputs


In [ ]:
MANDATORY=[
'experiment_identity.json','source_v82_record.json','executive_summary.txt',
'tables/threshold_frontier_summary.csv','tables/threshold_frontier_by_page.csv','tables/threshold_frontier_by_source_group.csv',
'tables/incremental_threshold_band_summary.csv','tables/incremental_threshold_pixels.parquet',
'tables/incremental_component_review_queue_all.csv','tables/catastrophic_components_all_thresholds.csv',
'tables/catastrophic_component_first_appearance_inventory.csv','tables/computable_gate_map.csv',
'tables/representative_page_selection.csv','audit/whole_page_integrity_and_hashes.csv','upload/upload_manifest.csv']

def completed_runs(root):
    root=Path(root)
    if not root.exists():return []
    return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in MANDATORY[:8])],key=lambda p:p.stat().st_mtime)

NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else (completed_runs(NB37_ROOT)[-1] if completed_runs(NB37_ROOT) else None)
if NB37 is None: fail_closed('input_resolution','No complete-looking Notebook 37 run found',{'root':str(NB37_ROOT)})
rows=[]
for rel in MANDATORY:
    p=NB37/rel
    rows.append({'relative_path':rel,'exists':p.exists(),'bytes':p.stat().st_size if p.is_file() else np.nan,'sha256':sha256_file(p) if p.is_file() else ''})
input_audit=pd.DataFrame(rows); atomic_csv(input_audit,AUDIT/'input_resolution_audit.csv')
missing=input_audit.loc[~input_audit.exists,'relative_path'].tolist()
if missing: fail_closed('input_resolution','Mandatory Notebook 37 inputs are missing',{'missing':missing,'run':str(NB37)})
print('Mandatory Notebook 37 inputs resolved:',len(input_audit))


## 4. Amended Notebook 37 upload-manifest audit


In [ ]:
manifest=pd.read_csv(NB37/'upload/upload_manifest.csv')
path_col=find_col(manifest,['relative_path','path','file','filename'])
hash_col=find_col(manifest,['sha256','sha_256','hash'],False)
mandatory_abs={str((NB37/r).resolve()):r for r in MANDATORY}

def manifest_candidates(raw):
    text=str(raw).strip().replace('\\','/')
    q=Path(text); out=[]
    if q.is_absolute():out.append(q)
    out.extend([NB37/q,NB37/'upload'/q])
    parts=list(q.parts)
    if 'upload' in parts:
        k=parts.index('upload'); tail=Path(*parts[k+1:]) if k+1<len(parts) else Path()
        out.extend([NB37/'upload'/tail,NB37/tail])
    if q.name:
        hits=[x for x in NB37.rglob(q.name) if x.is_file() and (x.as_posix().endswith(text) or x.name==q.name)]
        if len({str(x.resolve()) for x in hits})==1:out.extend(hits)
    dedup=[];seen=set()
    for x in out:
        key=str(x)
        if key not in seen:dedup.append(x);seen.add(key)
    return dedup

checks=[]
for _,r in manifest.iterrows():
    raw=str(r[path_col]); resolved=next((x for x in manifest_candidates(raw) if x.is_file()),None)
    actual=sha256_file(resolved) if resolved else ''; expected=str(r[hash_col]).strip() if hash_col and pd.notna(r[hash_col]) else ''
    required_rel=mandatory_abs.get(str(resolved.resolve())) if resolved else None
    match=(actual.lower()==expected.lower()) if expected and resolved else np.nan
    status=('verified' if resolved and (not expected or match is True) else 'required_hash_mismatch' if required_rel and match is False else 'package_row_hash_mismatch' if resolved and match is False else 'unresolved_package_row')
    checks.append({'manifest_path':raw,'resolved':str(resolved or ''),'required_by_notebook38':bool(required_rel),'required_relative_path':required_rel or '','exists':bool(resolved),'expected_sha256':expected,'actual_sha256':actual,'hash_matches':match,'status':status})
manifest_audit=pd.DataFrame(checks); atomic_csv(manifest_audit,AUDIT/'nb37_manifest_verification.csv')
required_bad=manifest_audit.required_by_notebook38 & manifest_audit.status.ne('verified')
if required_bad.any(): fail_closed('manifest_verification','A mandatory Notebook 38 source artefact failed Notebook 37 manifest verification',{'rows':manifest_audit.loc[required_bad].to_dict('records')})
manifest_summary={'rows':int(len(manifest_audit)),'verified':int(manifest_audit.status.eq('verified').sum()),'unresolved_package_rows':int(manifest_audit.status.eq('unresolved_package_row').sum()),'package_row_hash_mismatches':int(manifest_audit.status.eq('package_row_hash_mismatch').sum()),'mandatory_source_failures':int(required_bad.sum()),'interpretation':'Package-layout warnings do not invalidate directly resolved mandatory analytical inputs.'}
atomic_json(AUDIT/'nb37_manifest_verification_summary.json',manifest_summary); print(json.dumps(manifest_summary,indent=2))


## 5. Threshold identity, source lock and registered D3–D8 replay


In [ ]:
frontier=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv')
actual=[round(float(x),3) for x in frontier[find_col(frontier,['threshold'])]]
if actual!=THRESHOLDS: fail_closed('threshold_identity','Notebook 37 threshold order differs from protocol',{'expected':THRESHOLDS,'actual':actual})
LOCK.update({'source_nb37_run':str(NB37),'source_identity_sha256':sha256_file(NB37/'experiment_identity.json'),'source_v82_record_sha256':sha256_file(NB37/'source_v82_record.json'),'source_selection':'explicit_override' if SOURCE_NB37_RUN_OVERRIDE else 'latest_complete_looking'})
atomic_json(OUT/'protocol_lock.json',LOCK); shutil.copy2(NB37/'experiment_identity.json',REF/'notebook37_experiment_identity.json'); shutil.copy2(NB37/'source_v82_record.json',REF/'source_v82_record.json')
needed=['threshold','true_print','false_annotation','truth_uncertain_print','selected_print_pixels','utility','catastrophic_components','false_true_ratio','false_preservation_rate','wilson95_upper_false_preservation','bootstrap_utility_95_low','bootstrap_false_true_95_high','benefiting_pages','benefiting_source_groups','source_stability_for_groups_with_3_pages','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability','core_development_envelope_D3_to_D8']
miss=[x for x in needed if x not in frontier]
if miss:fail_closed('registered_gate_replay','Missing registered fields',{'columns':miss})
replay=pd.DataFrame({'threshold':frontier.threshold,'D3':frontier.catastrophic_components.eq(0),'D4':frontier.false_true_ratio.le(.10)&frontier.bootstrap_false_true_95_high.le(.20),'D5':frontier.false_preservation_rate.le(.005)&frontier.wilson95_upper_false_preservation.le(.01),'D6':frontier.utility.gt(0)&frontier.bootstrap_utility_95_low.gt(0),'D7':frontier.true_print.ge(500)&frontier.benefiting_pages.ge(10)&frontier.benefiting_source_groups.ge(3),'D8':frontier.source_stability_for_groups_with_3_pages.astype(bool)})
for i,s in enumerate(['D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'],3):
    if not np.array_equal(replay[f'D{i}'].to_numpy(),frontier[s].astype(bool).to_numpy()):fail_closed('registered_gate_replay',f'D{i} replay mismatch')
atomic_csv(frontier[needed],TABLES/'registered_D3_D8_results.csv'); atomic_csv(replay,AUDIT/'registered_gate_replay.csv'); print('Registered D3–D8 replay verified.')


## 6. Construct stable threshold-transition review units


In [ ]:
queue=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv'); cat_first=pd.read_csv(NB37/'tables/catastrophic_component_first_appearance_inventory.csv')
alias={'page':['page'],'source_group':['source_group'],'mask_component_id':['mask_component_id','component_id'],'higher_threshold':['higher_threshold','from_threshold','high_threshold'],'lower_threshold':['lower_threshold','to_threshold','low_threshold'],'new_selected':['new_selected','new_selected_pixels','incremental_selected','selected'],'new_true_print':['new_true_print','incremental_true_print','true_print'],'new_false_annotation':['new_false_annotation','incremental_false_annotation','false_annotation'],'new_truth_uncertain':['new_truth_uncertain','incremental_truth_uncertain','truth_uncertain'],'local_panel':['local_panel','local_crop','local_path','local_crop_path'],'wide_panel':['wide_panel','wide_crop','wide_path','wide_crop_path'],'locator':['locator','whole_page_locator','locator_path','whole_page_locator_path'],'truth_overlay':['truth_overlay','cumulative_truth_overlay','incremental_truth_overlay','truth_overlay_path']}
units=pd.DataFrame()
for target,names in alias.items():
    c=find_col(queue,names,target in ['page','source_group','mask_component_id','higher_threshold','lower_threshold'])
    if c is not None:units[target]=queue[c]
for c in ['new_selected','new_true_print','new_false_annotation','new_truth_uncertain']:
    if c not in units:units[c]=0
    units[c]=pd.to_numeric(units[c],errors='coerce').fillna(0).astype(int)
units.page=units.page.astype(str);units.source_group=units.source_group.astype(str);units.mask_component_id=pd.to_numeric(units.mask_component_id).astype(int);units.higher_threshold=pd.to_numeric(units.higher_threshold).round(3);units.lower_threshold=pd.to_numeric(units.lower_threshold).round(3)
cp=find_col(cat_first,['page']);cc=find_col(cat_first,['mask_component_id','component_id']);ct=find_col(cat_first,['first_appearance_threshold','threshold','lower_threshold'])
cf=cat_first[[cp,cc,ct]].copy();cf.columns=['page','mask_component_id','first_appearance_threshold'];cf.page=cf.page.astype(str);cf.mask_component_id=pd.to_numeric(cf.mask_component_id).astype(int);cf.registered_catastrophic=True
units=units.merge(cf,on=['page','mask_component_id'],how='left');units.registered_catastrophic=units.registered_catastrophic.fillna(False).astype(bool)
units['lineage_key']=units.apply(lambda r:f'{r.page}|{r.source_group}|{r.mask_component_id}|{r.higher_threshold:.3f}|{r.lower_threshold:.3f}',axis=1)
units['review_unit_id']=units.lineage_key.map(lambda x:'RU_'+hashlib.sha256(f'{ASSIGNMENT_SEED}|{x}'.encode()).hexdigest()[:16])
if units.review_unit_id.duplicated().any():fail_closed('review_units','Duplicate stable review units')
def resolve_asset(v):
    if pd.isna(v) or str(v).strip()=='':return ''
    p=Path(str(v))
    for z in [p,NB37/p,NB37/'review'/p,NB37/'visual_audit'/p]:
        if z.is_file():return str(z)
    return ''
for c in ['local_panel','wide_panel','locator','truth_overlay']:
    if c not in units:units[c]=''
    else:units[c]=units[c].map(resolve_asset)
print('Review units:',len(units),'source groups:',units.source_group.nunique(),'registered catastrophes:',int(units.registered_catastrophic.sum()))


## 7. Source-group-separated calibration and audit reserve


In [ ]:
groups=units.groupby('source_group',as_index=False).agg(review_units=('review_unit_id','size'),catastrophic_units=('registered_catastrophic','sum'),pages=('page','nunique'))
rng=np.random.default_rng(ASSIGNMENT_SEED);groups['_tie']=rng.random(len(groups));groups=groups.sort_values(['catastrophic_units','review_units','_tie'],ascending=[False,False,True])
target=round(len(units)*CALIBRATION_GROUP_FRACTION);cal_groups=[];audit_groups=[];n=0
for _,r in groups.iterrows():
    if n<target:cal_groups.append(r.source_group);n+=int(r.review_units)
    else:audit_groups.append(r.source_group)
if len(groups)>=2 and (not cal_groups or not audit_groups):cal_groups=[groups.iloc[0].source_group];audit_groups=groups.iloc[1:].source_group.tolist()
if set(cal_groups)&set(audit_groups):fail_closed('review_split','Source-group overlap between calibration and audit reserve')
groups['assignment_split']=np.where(groups.source_group.isin(cal_groups),'rubric_calibration','audit_reserve');groups=groups.drop(columns='_tie')
units=units.merge(groups[['source_group','assignment_split']],on='source_group',validate='many_to_one');atomic_csv(groups,TABLES/'review_split_map.csv');atomic_csv(units,TABLES/'review_unit_manifest.csv');print(groups.to_string(index=False))


## 8. Resolve adjacent-threshold whole-page pairs and blind A/B placement


In [ ]:
def threshold_tokens(t):return [f'{t:.3f}',f'{t:.3f}'.replace('.',''),f'{t:.3f}'.replace('.','_'),str(float(t))]
def resolve_whole(page,t):
    d=NB37/'whole_pages'/str(page)
    if not d.exists():return None
    scored=[]
    for p in d.rglob('*'):
        if p.suffix.lower() not in ['.png','.jpg','.jpeg']:continue
        s=sum(tok in p.name.lower() for tok in threshold_tokens(t))-3*sum(x in p.name.lower() for x in ['difference','overlay','decision','locator'])
        if float(t)==.995 and ('registered' in p.name.lower() or 'v8_2' in p.name.lower()):s+=2
        if s>0:scored.append((s,p))
    return max(scored,key=lambda z:z[0])[1] if scored else None
def resolve_observed(page):
    d=NB37/'whole_pages'/str(page)
    for pat in ['observed.png','*observed*.png','*annotated*.png']:
        hits=list(d.glob(pat)) if d.exists() else []
        if hits:return str(hits[0])
    return ''
rows=[];missing=[]
for _,r in units.iterrows():
    hi=resolve_whole(r.page,r.higher_threshold);lo=resolve_whole(r.page,r.lower_threshold)
    if not hi or not lo or not valid_image(hi) or not valid_image(lo):missing.append({'review_unit_id':r.review_unit_id,'page':r.page,'higher_path':str(hi or ''),'lower_path':str(lo or '')});continue
    left_low=int(hashlib.sha256(f'{ASSIGNMENT_SEED}|{r.review_unit_id}'.encode()).hexdigest(),16)%2==0
    A,B=(lo,hi) if left_low else (hi,lo);mapping={'A':'lower','B':'higher'} if left_low else {'A':'higher','B':'lower'}
    rows.append({**r.to_dict(),'whole_page_A':str(A),'whole_page_B':str(B),'A_sha256':sha256_file(A),'B_sha256':sha256_file(B),'observed_path':resolve_observed(r.page),'randomisation_side_locked':base64.urlsafe_b64encode(json.dumps(mapping).encode()).decode()})
atomic_csv(pd.DataFrame(missing),AUDIT/'missing_review_assets.csv')
if missing:fail_closed('review_assets','Unreadable or missing whole-page pairs',{'count':len(missing),'examples':missing[:10]})
blind=pd.DataFrame(rows);atomic_csv(blind,TABLES/'blinded_review_manifest.csv');print('Blinded review units:',len(blind))


## 9. Gradio single-reviewer interface

First pass shows only randomised whole pages and locks the response. Available wide/local/locator, observed and truth-overlay diagnostics appear only after the first-pass save. Diagnostic ratings are separate records and never overwrite round one.


In [ ]:
import gradio as gr
R1=REVIEW/'perceptual_review_round1.csv';R2=REVIEW/'diagnostic_review_round2.csv'
R1C=['review_unit_id','reviewer_id','initial_visibility','pair_preference','harm_severity','benefit_severity','defect_scope','musical_function_at_risk','annotation_prominence','reviewer_action','confidence','review_note','timestamp_utc','locked_before_diagnostic','record_sha256']
R2C=['review_unit_id','reviewer_id','diagnostic_mechanism','resolved_harm_severity','resolved_benefit_severity','diagnostic_note','timestamp_utc','record_sha256']
def load_records(path,cols):
    if not Path(path).exists():return pd.DataFrame(columns=cols)
    d=pd.read_csv(path,keep_default_na=False)
    for c in cols:
        if c not in d:d[c]=''
    return d[cols]
def save_record(path,cols,d):
    d['record_sha256']=record_hash(d);z=load_records(path,cols);z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])];atomic_csv(pd.concat([z,pd.DataFrame([d])],ignore_index=True)[cols],path);return d['record_sha256']
def pending_indices():
    done=set(load_records(R2,R2C).review_unit_id.astype(str));return [i for i,x in enumerate(blind.review_unit_id.astype(str)) if x not in done]
def img(x):return str(x) if str(x) and Path(str(x)).is_file() else None
def load_view(i):
    i=int(max(0,min(i,len(blind)-1)));r=blind.iloc[i];n=len(load_records(R2,R2C));return i,img(r.whole_page_A),img(r.whole_page_B),f'Item {i+1}/{len(blind)} — complete {n}/{len(blind)} — single-reviewer exploratory',r.review_unit_id
def save_first(i,vis,pref,harm,benefit,scope,musical,prom,action,conf,note):
    r=blind.iloc[int(i)];vals=[vis,pref,harm,benefit,scope,musical,prom,action,conf]
    if any(v in [None,''] for v in vals):raise gr.Error('Complete all first-pass fields.')
    if (harm in ['P2','P3','P4','U'] or benefit in ['B3','B4','U']) and not str(note).strip():raise gr.Error('A note is required for P2–P4, B3–B4 or U.')
    d={'review_unit_id':r.review_unit_id,'reviewer_id':REVIEWER_ID,'initial_visibility':vis,'pair_preference':pref,'harm_severity':harm,'benefit_severity':benefit,'defect_scope':scope,'musical_function_at_risk':musical,'annotation_prominence':prom,'reviewer_action':action,'confidence':conf,'review_note':str(note).strip(),'timestamp_utc':utc_now(),'locked_before_diagnostic':True};h=save_record(R1,R1C,d)
    return f'First pass locked — {h[:12]}',gr.update(visible=False),gr.update(visible=True),img(r.wide_panel),img(r.local_panel),img(r.locator),img(r.observed_path),img(r.truth_overlay)
def save_diag(i,mechanism,harm,benefit,note):
    r=blind.iloc[int(i)]
    if not mechanism or not harm or not benefit:raise gr.Error('Complete diagnostic mechanism and resolved ratings.')
    d={'review_unit_id':r.review_unit_id,'reviewer_id':REVIEWER_ID,'diagnostic_mechanism':mechanism,'resolved_harm_severity':harm,'resolved_benefit_severity':benefit,'diagnostic_note':str(note).strip(),'timestamp_utc':utc_now()};h=save_record(R2,R2C,d)
    pending=pending_indices();nxt=next((x for x in pending if x>int(i)),pending[0] if pending else int(i));v=load_view(nxt)
    return *v,f'Diagnostic saved — {h[:12]}',gr.update(visible=True),gr.update(visible=False)

with gr.Blocks(title='Notebook 38 Review') as app:
    gr.Markdown('# Notebook 38 — blinded perceptual review')
    state=gr.State(pending_indices()[0] if pending_indices() else 0);unit=gr.Textbox(label='Opaque review unit',interactive=False);status=gr.Markdown()
    with gr.Row():A=gr.Image(label='A',type='filepath',height=700);B=gr.Image(label='B',type='filepath',height=700)
    with gr.Column(visible=True) as panel1:
        vis=gr.Radio(['no difference','visible difference','unclear'],label='Initial visibility');pref=gr.Radio(['A','B','neither','mixed','unclear'],label='Pair preference')
        with gr.Row():harm=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Harm');benefit=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Benefit')
        scope=gr.Dropdown(['pixel-fragment','component-local','staff/system','page-repeated','page-global'],label='Defect scope');musical=gr.Dropdown(['none','staff','notehead','stem/beam','accidental','rest','articulation','dynamic','text/lyric','barline/repeat','other','unclear'],label='Musical function at risk');prom=gr.Radio(['none','faint','moderate','prominent'],label='Annotation prominence');action=gr.Radio(['accept','accept-with-note','manual-correct','reject-policy-case','unclear'],label='Reviewer action');conf=gr.Radio(['low','medium','high'],label='Confidence');note1=gr.Textbox(label='First-pass note',lines=3);button1=gr.Button('Lock first pass and reveal diagnostics',variant='primary')
    with gr.Column(visible=False) as panel2:
        gr.Markdown('## Diagnostics — first pass locked')
        with gr.Row():wide=gr.Image(label='Wide');local=gr.Image(label='Local');locator=gr.Image(label='Locator');observed=gr.Image(label='Observed');truth=gr.Image(label='Truth/overlay diagnostic')
        mechanism=gr.Dropdown(['correct print recovery','retained annotation','mixed overlap','registration ambiguity','truth-boundary sensitivity','display artefact','other'],label='Mechanism')
        with gr.Row():rh=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Resolved harm');rb=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Resolved benefit')
        note2=gr.Textbox(label='Diagnostic note',lines=3);button2=gr.Button('Save diagnostic and load next',variant='primary')
    message=gr.Markdown();app.load(load_view,[state],[state,A,B,status,unit]);button1.click(save_first,[state,vis,pref,harm,benefit,scope,musical,prom,action,conf,note1],[message,panel1,panel2,wide,local,locator,observed,truth]);button2.click(save_diag,[state,mechanism,rh,rb,note2],[state,A,B,status,unit,message,panel1,panel2])
app.launch(share=True,debug=False)


## 10. Review-completion gate and severity-map lock


In [ ]:
r1=load_records(R1,R1C);r2=load_records(R2,R2C);expected=set(blind.review_unit_id.astype(str));a=set(r1.review_unit_id.astype(str));b=set(r2.review_unit_id.astype(str))
coverage=pd.DataFrame({'review_unit_id':sorted(expected),'round1_complete':[x in a for x in sorted(expected)],'round2_complete':[x in b for x in sorted(expected)]});atomic_csv(coverage,AUDIT/'review_coverage.csv')
if not coverage[['round1_complete','round2_complete']].all().all():fail_closed('review_completion','Mandatory review coverage is incomplete',{'expected':len(expected),'round1':len(a&expected),'round2':len(b&expected)})
if not r1.locked_before_diagnostic.astype(str).str.lower().isin(['true','1']).all():fail_closed('review_completion','First-pass lock violation')
atomic_csv(pd.DataFrame([{'reviewer_count':1,'raw_agreement':np.nan,'exact_agreement':np.nan,'within_one_level_agreement':np.nan,'status':'not_estimable_single_reviewer'}]),TABLES/'reviewer_agreement.csv')
joined=blind.merge(r1,on='review_unit_id',validate='one_to_one').merge(r2,on=['review_unit_id','reviewer_id'],validate='one_to_one')
cal=joined[joined.assignment_split.eq('rubric_calibration')].copy();audit_reviews=joined[joined.assignment_split.eq('audit_reserve')].copy()
if cal.empty or audit_reviews.empty:fail_closed('review_split','Empty calibration or audit reserve')
severity_lock={'protocol_version':PROTOCOL_VERSION,'frozen_utc':utc_now(),'calibration_source_groups':sorted(cal.source_group.unique()),'calibration_units':len(cal),'calibration_manifest_sha256':hashlib.sha256(cal[['review_unit_id','resolved_harm_severity','resolved_benefit_severity']].to_csv(index=False).encode()).hexdigest(),'weights':LOCK['D6_weights'],'mapping_tuned_to_audit_reserve':False,'single_reviewer_exploratory':True}
atomic_json(OUT/'severity_mapping_lock.json',severity_lock);print(json.dumps(severity_lock,indent=2))


## 11. Audit-reserve D3–D8 sensitivity analysis


In [ ]:
harm_num={'P0':0,'P1':1,'P2':2,'P3':3,'P4':4};benefit_num={'B0':0,'B1':1,'B2':2,'B3':3,'B4':4}
audit_reviews['harm_n']=audit_reviews.resolved_harm_severity.map(harm_num);audit_reviews['benefit_n']=audit_reviews.resolved_benefit_severity.map(benefit_num)
if audit_reviews[['harm_n','benefit_n']].isna().any().any():fail_closed('analysis','Unresolved U remains in audit reserve')
by_page=pd.read_csv(NB37/'tables/threshold_frontier_by_page.csv');by_source=pd.read_csv(NB37/'tables/threshold_frontier_by_source_group.csv')
audit_group_names=set(groups.loc[groups.assignment_split.eq('audit_reserve'),'source_group'].astype(str));pages_all=by_page[by_page.source_group.astype(str).isin(audit_group_names)].copy();sources_all=by_source[by_source.source_group.astype(str).isin(audit_group_names)].copy()
def active(t):return audit_reviews[audit_reviews.lower_threshold.astype(float).ge(t)].copy()
def wilson(e,n,z=1.96):
    if n<=0:return np.nan
    p=e/n;d=1+z*z/n;return (p+z*z/(2*n)+z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/d
def bootstrap(pages,seed):
    vals=pages[['true_print','false_annotation','truth_uncertain_print','catastrophic_components']].to_numpy(float);rng=np.random.default_rng(seed);out=[];remaining=BOOTSTRAP_RESAMPLES
    while remaining:
        size=min(1000,remaining);tot=vals[rng.integers(0,len(vals),size=(size,len(vals)))].sum(1);tp,fp,tu,c=tot.T;out.append(np.c_[tp-5*fp-500*c,fp/np.maximum(tp,1),fp/np.maximum(tp+fp+tu,1)]);remaining-=size
    return np.vstack(out)
P=[];U=[];D7=[];D8=[]
for t in THRESHOLDS:
    pages=pages_all[np.isclose(pages_all.threshold,t)];sources=sources_all[np.isclose(sources_all.threshold,t)];reviews=active(t)
    if pages.empty:continue
    tp=int(pages.true_print.sum());fp=int(pages.false_annotation.sum());tu=int(pages.truth_uncertain_print.sum());cats=int(pages.catastrophic_components.sum());selected=int(pages.selected.sum());boot=bootstrap(pages,BOOTSTRAP_SEED+round(1000*t))
    p2=reviews.harm_n.ge(2);p3=reviews.harm_n.ge(3);p4=reviews.harm_n.eq(4);repeated_p3=bool((reviews.loc[p3].groupby(['source_group','diagnostic_mechanism']).size()>=2).any()) if p3.any() else False
    P.append({'threshold':t,'true_print':tp,'false_annotation':fp,'truth_uncertain_print':tu,'selected':selected,'catastrophes':cats,'utility':tp-5*fp-500*cats,'utility_95_low':np.quantile(boot[:,0],.025),'false_true':fp/max(tp,1),'false_true_95_high':np.quantile(boot[:,1],.975),'false_preservation':fp/max(selected,1),'false_preservation_wilson_upper':wilson(fp,selected),'P2_plus':int(p2.sum()),'P3_plus':int(p3.sum()),'P4':int(p4.sum()),'repeated_P3':repeated_p3,'D3_R0':cats==0,'D3_R1':bool((~reviews.registered_catastrophic|reviews.harm_n.le(1)).all() and not p2.any()),'D3_R2':bool(not p3.any() and not ((reviews.loc[p2].groupby(['source_group','diagnostic_mechanism']).size()>=2).any() if p2.any() else False)),'D3_R3':not p3.any()})
    counts=reviews.resolved_harm_severity.value_counts().to_dict()
    for w,a1,a2,a3 in itertools.product(FP_WEIGHTS,P1_WEIGHTS,P2_WEIGHTS,P3_WEIGHTS):
        penalty=counts.get('P1',0)*a1+counts.get('P2',0)*a2+counts.get('P3',0)*a3;U.append({'threshold':t,'FP_weight':w,'P1_weight':a1,'P2_weight':a2,'P3_weight':a3,'P4_hard_fail':bool(p4.any()),'severity_adjusted_utility':tp-w*fp-penalty})
    for label,minb in [('B1+',1),('B2+',2),('B3+',3)]:
        z=reviews[reviews.harm_n.le(1)&reviews.benefit_n.ge(minb)];ztp=int(z.new_true_print.sum());zp=z.loc[z.new_true_print.gt(0),'page'].nunique();zg=z.loc[z.new_true_print.gt(0),'source_group'].nunique()
        for req_tp,req_p,req_g in itertools.product(D7_TP,D7_PAGES,D7_GROUPS):D7.append({'threshold':t,'benefit':label,'qualified_tp':ztp,'qualified_pages':zp,'qualified_groups':zg,'required_tp':req_tp,'required_pages':req_p,'required_groups':req_g,'passes':ztp>=req_tp and zp>=req_p and zg>=req_g and not p4.any()})
    for _,x in sources.iterrows():
        rr=reviews[reviews.source_group.astype(str).eq(str(x.source_group))];q3=int(rr.harm_n.ge(3).sum());q4=int(rr.harm_n.eq(4).sum());repeat=bool((rr.loc[rr.harm_n.eq(3)].groupby('diagnostic_mechanism').size()>=2).any()) if q3 else False
        D8.append({'threshold':t,'source_group':x.source_group,'pages':int(x.pages),'false_true_ratio':float(x.false_true_ratio),'catastrophes':int(x.catastrophic_components),'utility':float(x.utility),'P3':q3,'P4':q4,'D8_R0':int(x.pages)<3 or (float(x.false_true_ratio)<=.25 and int(x.catastrophic_components)==0),'D8_R1':float(x.false_true_ratio)<=.35 and q3==0 and float(x.utility)>=0,'D8_R2':float(x.false_true_ratio)<=.50 and q4==0 and not repeat,'D8_R3_source_scoped_only':True})
perceptual=pd.DataFrame(P);utility_matrix=pd.DataFrame(U);d7=pd.DataFrame(D7);d8=pd.DataFrame(D8)
for df,name in [(perceptual,'perceptual_D3_D8_results.csv'),(perceptual,'audit_reserve_envelope_results.csv'),(utility_matrix,'severity_adjusted_utility_matrix.csv'),(d7,'D7_sensitivity_matrix.csv'),(d8,'D8_source_stability_variants.csv'),(pages_all,'results_by_page.csv'),(sources_all,'results_by_source_group.csv')]:atomic_csv(df,TABLES/name)
print(perceptual.to_string(index=False))


## 12. Review burden, leave-one-group-out and relaxation frontier


In [ ]:
burden=[];logo=[];front=[]
for t in THRESHOLDS:
    r=active(t);p=pages_all[np.isclose(pages_all.threshold,t)]
    if len(r):burden.append({'threshold':t,'review_units':len(r),'P2_plus':int(r.harm_n.ge(2).sum()),'P3_plus':int(r.harm_n.ge(3).sum()),'P4':int(r.harm_n.eq(4).sum()),'review_burden_rate':float(r.harm_n.ge(2).mean())})
    for omitted in sorted(audit_group_names):
        z=p[~p.source_group.astype(str).eq(omitted)]
        if len(z):
            tp=int(z.true_print.sum());fp=int(z.false_annotation.sum());c=int(z.catastrophic_components.sum());logo.append({'threshold':t,'omitted_source_group':omitted,'true_print':tp,'false_annotation':fp,'catastrophes':c,'utility':tp-5*fp-500*c,'false_true':fp/max(tp,1)})
for _,r in perceptual.sort_values('threshold',ascending=False).iterrows():
    z=d8[np.isclose(d8.threshold,r.threshold)];front.append({'threshold':r.threshold,'true_print':r.true_print,'false_annotation':r.false_annotation,'P2_plus':r.P2_plus,'P3_plus':r.P3_plus,'P4':r.P4,'D3_R1':r.D3_R1,'D3_R2':r.D3_R2,'D3_R3':r.D3_R3,'D8_R1_all':bool(z.D8_R1.all()),'D8_R2_all':bool(z.D8_R2.all()),'hard_stop':bool(r.P4>0 or r.repeated_P3),'exploratory_only':True})
burden=pd.DataFrame(burden);logo=pd.DataFrame(logo);relax=pd.DataFrame(front);atomic_csv(burden,TABLES/'review_burden_summary.csv');atomic_csv(logo,TABLES/'leave_one_group_out.csv');atomic_csv(relax,TABLES/'envelope_relaxation_frontier.csv');print(relax.to_string(index=False))


## 13. Visual summaries and exploratory verdict


In [ ]:
plt.figure(figsize=(8,5));plt.plot(perceptual.false_annotation,perceptual.true_print,marker='o')
for _,r in perceptual.iterrows():plt.annotate(f'{r.threshold:.3f}',(r.false_annotation,r.true_print),fontsize=8)
plt.xlabel('False-annotation pixels');plt.ylabel('True-print pixels');plt.title('Audit-reserve benefit versus truth error');plt.tight_layout();plt.savefig(VISUALS/'perceptual_frontier.png',dpi=160);plt.close()
severity=[]
for t in THRESHOLDS:
    for s,n in active(t).resolved_harm_severity.value_counts().items():severity.append({'threshold':t,'severity':s,'count':n})
if severity:
    pd.DataFrame(severity).pivot(index='threshold',columns='severity',values='count').fillna(0).sort_index(ascending=False).plot(kind='bar',stacked=True,figsize=(10,5));plt.tight_layout();plt.savefig(VISUALS/'severity_by_threshold.png',dpi=160);plt.close()
stop=relax[relax.hard_stop].head(1);first_stop=float(stop.iloc[0].threshold) if len(stop) else None
outcome='reject_relaxation_at_or_below_first_P4' if (perceptual.P4>0).any() else ('reject_general_relaxation_at_or_below_repeated_P3' if perceptual.repeated_P3.any() else ('potential_supervised_review_tier_exploratory' if (perceptual.P2_plus>0).any() else 'no_material_harm_observed_exploratory_only'))
verdict={'notebook_version':NOTEBOOK_VERSION,'status':'single-reviewer-exploratory','outcome':outcome,'first_material_harm_or_instability_threshold':first_stop,'may_nominate_revised_envelope':False,'production_authority':False,'threshold_adoption_authority':False,'untouched_future_validation_used':False,'D9':'not_validated_training_role_sparse_material_only'};atomic_json(OUT/'notebook38_verdict.json',verdict)
summary='\n'.join(['Notebook 38 — V8.2i amended','Status: SINGLE-REVIEWER EXPLORATORY','Outcome: '+outcome,'First material-harm/instability threshold: '+str(first_stop),f'Review units: {len(joined)}; calibration: {len(cal)}; audit reserve: {len(audit_reviews)}','D9 sparse/hard-negative: not validated; existing material served a sparse training role.','Authority: no threshold adoption, independent-test nomination, or production use.'])+"\n";atomic_text(OUT/'executive_summary.txt',summary)
decision='\n'.join(['# Notebook 38 Decision Record','', '- Status: single-reviewer exploratory.',f'- Outcome: `{outcome}`.',f'- First hard stop: `{first_stop}`.','- Registered D3–D8 retained.','- Agreement is not estimable with one reviewer.','- No threshold adoption or independent-test nomination.','']);atomic_text(OUT/'decision_record.md',decision);print(summary)


## 14. Curated final upload package


In [ ]:
items=[OUT/'protocol_lock.json',OUT/'severity_mapping_lock.json',OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',OUT/'decision_record.md',TABLES/'review_unit_manifest.csv',TABLES/'review_split_map.csv',TABLES/'blinded_review_manifest.csv',R1,R2,TABLES/'reviewer_agreement.csv',TABLES/'registered_D3_D8_results.csv',TABLES/'perceptual_D3_D8_results.csv',TABLES/'audit_reserve_envelope_results.csv',TABLES/'envelope_relaxation_frontier.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',TABLES/'leave_one_group_out.csv',TABLES/'review_burden_summary.csv',TABLES/'severity_adjusted_utility_matrix.csv',TABLES/'D7_sensitivity_matrix.csv',TABLES/'D8_source_stability_variants.csv',VISUALS/'perceptual_frontier.png',VISUALS/'severity_by_threshold.png',AUDIT/'input_resolution_audit.csv',AUDIT/'nb37_manifest_verification.csv',AUDIT/'nb37_manifest_verification_summary.json',AUDIT/'registered_gate_replay.csv',AUDIT/'review_coverage.csv',AUDIT/'missing_review_assets.csv',REF/'notebook37_experiment_identity.json',REF/'source_v82_record.json']
missing=[str(x.relative_to(OUT)) for x in items if not x.exists()]
if missing:fail_closed('final_package','Missing curated outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
    rel=src.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
readme='Notebook 38 upload package. Single-reviewer exploratory: no threshold adoption, independent-test nomination or production authority.\n';atomic_text(UPLOAD/'README.txt',readme);rows.append({'relative_path':'README.txt','bytes':(UPLOAD/'README.txt').stat().st_size,'sha256':sha256_file(UPLOAD/'README.txt')})
files=sorted(x.relative_to(UPLOAD).as_posix() for x in UPLOAD.rglob('*') if x.is_file());atomic_text(UPLOAD/'upload_file_list.txt','\n'.join(files)+'\n');rows.append({'relative_path':'upload_file_list.txt','bytes':(UPLOAD/'upload_file_list.txt').stat().st_size,'sha256':sha256_file(UPLOAD/'upload_file_list.txt')});atomic_csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv')
for _,r in pd.read_csv(UPLOAD/'upload_manifest.csv').iterrows():
    if sha256_file(UPLOAD/r.relative_path)!=r.sha256:fail_closed('final_package','Hash verification failed',{'file':r.relative_path})
zip_path=OUT/'notebook38_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
atomic_json(OUT/'final_package_record.json',{'zip':str(zip_path),'bytes':zip_path.stat().st_size,'sha256':sha256_file(zip_path),'utc':utc_now()});print('Final bundle:',zip_path)


## Interpretation boundary

A completed run may describe perceptually negligible registered alerts, review burden, source sensitivity and the first material-harm transition. With one reviewer it remains exploratory and does not nominate a revised envelope. Any candidate must be separately frozen and tested on fresh untouched evidence.
